In [52]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer

In [53]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [54]:
data=data.drop(columns=['RowNumber','CustomerId','Surname'],axis=1)

In [55]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [56]:
label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])

In [57]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [58]:
preprocessor=ColumnTransformer(
    transformers=[
        ('onehot',OneHotEncoder(drop='first'),['Geography']),
        ('scaler',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts'])
    ],
    remainder='passthrough'
)

In [59]:
X=data.drop(columns=['EstimatedSalary'])
y=data['EstimatedSalary']

In [60]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [61]:
X_train_scaled=preprocessor.fit_transform(X_train)

In [62]:
X_test_scaled=preprocessor.transform(X_test)

In [63]:
import pickle

In [64]:
with open('labelencoderregression.pkl','wb') as file:
    pickle.dump(label_encoder,file)


with open('preprocessor_regression.pkl','wb') as file:
    pickle.dump(preprocessor,file)    

ANN REGRESSION IMPLEMENTATION

In [65]:
import tensorflow as tf 


In [66]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [67]:
#build the model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train_scaled.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1) #linear activation function applied  by default 
])

In [68]:
##Compile the model 
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [69]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 64)                768       
                                                                 
 dense_10 (Dense)            (None, 32)                2080      
                                                                 
 dense_11 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2881 (11.25 KB)
Trainable params: 2881 (11.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [70]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

import datetime

#set up Tensorboard 
log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensornoard_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [71]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [72]:
history = model.fit(
    X_train_scaled,y_train,
    validation_data=(X_test_scaled,y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensornoard_callback]
)

Epoch 1/100
250/250 [==============================] - 2s 3ms/step - loss: 100342.6484 - mae: 100342.6484 - val_loss: 98343.8672 - val_mae: 98343.8672
Epoch 2/100
250/250 [==============================] - 1s 2ms/step - loss: 98852.9688 - mae: 98852.9688 - val_loss: 95280.7344 - val_mae: 95280.7344
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 93633.5703 - mae: 93633.5703 - val_loss: 87768.8047 - val_mae: 87768.8047
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 84247.0781 - mae: 84247.0781 - val_loss: 76873.3438 - val_mae: 76873.3438
Epoch 5/100
250/250 [==============================] - 1s 2ms/step - loss: 72785.9375 - mae: 72785.9375 - val_loss: 65624.1406 - val_mae: 65624.1406
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 62775.0781 - mae: 62775.0781 - val_loss: 57600.8672 - val_mae: 57600.8672
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 56936.8672 - mae: 56936.8672 

In [73]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [75]:
%tensorboard --logdir regressionlogs/fit

In [76]:
print(log_dir)

regressionlogs/fit/20260618-210121


In [77]:
import os

for root, dirs, files in os.walk(log_dir):
    print(root)
    print(files)

regressionlogs/fit/20260618-210121
[]
regressionlogs/fit/20260618-210121\train
['events.out.tfevents.1781796681.OJASPC.57996.3.v2']
regressionlogs/fit/20260618-210121\validation
['events.out.tfevents.1781796682.OJASPC.57996.4.v2']


In [79]:
# evaluate the model on the test data
test_loss,test_mae=model.evaluate(X_test_scaled,y_test)
print(f'TEST MAE : {test_mae}')

63/63 [==============================] - 0s 2ms/step - loss: 49858.2461 - mae: 49858.2461
TEST MAE : 49858.24609375


In [80]:
model.save('regressionmodel.h5')

c:\Users\ojass\OneDrive\Desktop\Udemy\DeepLearning\ANN\venvDL\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
